# 01 – Explore QB dropbacks

Loads cached play-by-play data (2016–present) and filters it to QB dropbacks. Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [1]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp

pl.Config.set_tbl_cols(30)

polars.config.Config

In [2]:
pbp = load_pbp()
pbp.shape

(500466, 372)

## Filter to QB dropbacks

`qb_dropback == 1` covers pass attempts, sacks, and QB scrambles.

In [3]:
dropbacks = pbp.filter(pl.col("qb_dropback") == 1)
dropbacks.shape

(217923, 372)

## Columns most relevant to rating QBs

| column | meaning |
|---|---|
| `season`, `week`, `season_type` | when the play happened (`REG` / `POST`) |
| `game_id`, `posteam`, `defteam` | game, offense, defense |
| `passer_player_id`, `passer_player_name` | the QB on the dropback |
| `epa` | expected points added by the play (our target is next-week EPA per dropback) |
| `qb_epa` | EPA, but not penalizing the QB for a receiver's fumble after the catch |
| `cpoe` | completion % over expected (pass attempts only; null otherwise) |
| `pass_attempt`, `complete_pass`, `sack`, `qb_scramble`, `interception` | play outcome flags |
| `air_yards`, `yards_after_catch`, `yards_gained` | yardage |
| `success` | 1 if EPA > 0 |
| `down`, `ydstogo`, `wp` | situation: down, distance, win probability |

In [4]:
qb_cols = [
    "season", "week", "season_type", "game_id", "posteam", "defteam",
    "passer_player_id", "passer_player_name",
    "qb_dropback", "epa", "qb_epa", "cpoe",
    "pass_attempt", "complete_pass", "sack", "qb_scramble", "interception",
    "air_yards", "yards_after_catch", "yards_gained", "success",
    "down", "ydstogo", "wp",
]
qb = dropbacks.select(qb_cols)
qb.head(10)

season,week,season_type,game_id,posteam,defteam,passer_player_id,passer_player_name,qb_dropback,epa,qb_epa,cpoe,pass_attempt,complete_pass,sack,qb_scramble,interception,air_yards,yards_after_catch,yards_gained,success,down,ydstogo,wp
i32,i32,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,0.64454,0.64454,13.441056,1.0,1.0,0.0,0.0,0.0,-4.0,9.0,5.0,1.0,2.0,4.0,0.541975
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,0.918295,0.918295,19.303071,1.0,1.0,0.0,0.0,0.0,1.0,8.0,9.0,1.0,2.0,10.0,0.52803
2016,1,"""REG""","""2016_01_BUF_BAL""","""BUF""","""BAL""","""00-0028118""","""T.Taylor""",1.0,-0.481404,-0.481404,20.712334,1.0,1.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,1.0,10.0,0.507005
2016,1,"""REG""","""2016_01_BUF_BAL""","""BUF""","""BAL""","""00-0028118""","""T.Taylor""",1.0,-1.374502,-1.374502,12.400633,1.0,1.0,0.0,0.0,0.0,-4.0,0.0,-4.0,0.0,3.0,14.0,0.447803
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,-0.18526,-0.18526,23.208666,1.0,1.0,0.0,0.0,0.0,3.0,0.0,3.0,0.0,1.0,10.0,0.56639
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,0.493441,0.493441,27.201819,1.0,1.0,0.0,0.0,0.0,5.0,1.0,6.0,1.0,2.0,7.0,0.559212
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,2.476346,2.476346,38.063121,1.0,1.0,0.0,0.0,0.0,16.0,0.0,16.0,1.0,3.0,11.0,0.510546
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,-0.456218,-0.456218,null,1.0,0.0,0.0,0.0,0.0,3.0,null,0.0,0.0,1.0,10.0,0.584447
2016,1,"""REG""","""2016_01_BUF_BAL""","""BUF""","""BAL""","""00-0028118""","""T.Taylor""",1.0,0.039256,0.039256,14.387792,1.0,1.0,0.0,0.0,0.0,-2.0,7.0,5.0,1.0,1.0,10.0,0.549582


In [5]:
qb.schema

Schema([('season', Int32),
        ('week', Int32),
        ('season_type', String),
        ('game_id', String),
        ('posteam', String),
        ('defteam', String),
        ('passer_player_id', String),
        ('passer_player_name', String),
        ('qb_dropback', Float64),
        ('epa', Float64),
        ('qb_epa', Float64),
        ('cpoe', Float64),
        ('pass_attempt', Float64),
        ('complete_pass', Float64),
        ('sack', Float64),
        ('qb_scramble', Float64),
        ('interception', Float64),
        ('air_yards', Float64),
        ('yards_after_catch', Float64),
        ('yards_gained', Float64),
        ('success', Float64),
        ('down', Float64),
        ('ydstogo', Float64),
        ('wp', Float64)])

# Phase 2: Explore + define the target

**Leakage guard:** every decision about the target is made by looking at the **training seasons only (2016–2023)**.
2024 (val) and 2025 (test) stay unseen, so their scores later are honest.

In [6]:
EXPLORE_SEASONS = list(range(2016, 2024))  # train only; do not widen this
train_pbp = pbp.filter(pl.col("season").is_in(EXPLORE_SEASONS))
train_pbp.select(pl.col("season").min().alias("first"), pl.col("season").max().alias("last"))

first,last
i32,i32
2016,2023


## Concept 1: one row per QB per game

Like turning a box of grocery receipts into one line per shopping trip: sort dropbacks into piles by
(game, team, passer), then write one summary line per pile with `dropbacks` and `epa_per_db`.
The code lives in `src/features/qb_games.py` so later phases reuse the exact same table.

In [7]:
from src.features.qb_games import qb_games

games = qb_games(train_pbp)

# How many team-games (one offense in one game) vs QB-games?
team_games = games.select(pl.struct("game_id", "posteam").n_unique()).item()
print(f"team-games: {team_games:,}   QB-games: {games.height:,}   ratio: {games.height / team_games:.2f}")

# How many different passers did each team-game have?
(games.group_by("game_id", "posteam").agg(pl.len().alias("passers"))
      .group_by("passers").agg(pl.len().alias("team_games")).sort("passers"))

team-games: 4,382   QB-games: 5,407   ratio: 1.23


passers,team_games
u32,u32
1,3431
2,880
3,68
4,3


### ▶ Play: Concept 1
Change a knob, rerun this cell, and compare against the default (`MIN_DROPBACKS = 1`).

In [10]:
# ---- KNOBS -----------------------------------------------------------------
MIN_DROPBACKS = 1                 # Try 3, then 10, then 20. Who disappears first? QBs or trick-play passers?
SHOW_GAME = "2023_01_DET_KC"      # Any game_id, format "<season>_<week 2 digits>_<AWAY>_<HOME>", e.g. "2022_10_SEA_TB"
# ---------------------------------------------------------------------------

kept = games.filter(pl.col("dropbacks") >= MIN_DROPBACKS)
dropped = games.filter(pl.col("dropbacks") < MIN_DROPBACKS)

print(f"Default (MIN_DROPBACKS=1): {games.height:,} rows")
print(f"Now (MIN_DROPBACKS={MIN_DROPBACKS}): {kept.height:,} rows  ->  dropped {dropped.height:,} "
      f"({dropped.height / games.height:.1%})")

if dropped.height:
    print("\nPassers who lost the most rows:")
    print(dropped.group_by("passer").agg(pl.len().alias("rows_dropped"), pl.col("dropbacks").sum().alias("dropbacks"))
                 .sort("rows_dropped", descending=True).head(10))

print(f"\nGame {SHOW_GAME} (rows marked kept=false would be dropped):")
games.filter(pl.col("game_id") == SHOW_GAME).with_columns(
    (pl.col("dropbacks") >= MIN_DROPBACKS).alias("kept")
).select("posteam", "passer", "dropbacks", "epa_sum", "epa_per_db", "kept")

Default (MIN_DROPBACKS=1): 5,407 rows
Now (MIN_DROPBACKS=20): 4,330 rows  ->  dropped 1,077 (19.9%)

Passers who lost the most rows:
shape: (10, 3)
┌───────────────┬──────────────┬───────────┐
│ passer        ┆ rows_dropped ┆ dropbacks │
│ ---           ┆ ---          ┆ ---       │
│ str           ┆ u32          ┆ u32       │
╞═══════════════╪══════════════╪═══════════╡
│ T.Hill        ┆ 54           ┆ 94        │
│ C.Rush        ┆ 16           ┆ 75        │
│ J.Brissett    ┆ 16           ┆ 65        │
│ J.Hekker      ┆ 16           ┆ 16        │
│ T.Taylor      ┆ 14           ┆ 122       │
│ P.Walker      ┆ 13           ┆ 142       │
│ B.Gabbert     ┆ 13           ┆ 90        │
│ B.Hoyer       ┆ 12           ┆ 71        │
│ J.Landry      ┆ 12           ┆ 12        │
│ R.Fitzpatrick ┆ 11           ┆ 141       │
└───────────────┴──────────────┴───────────┘

Game 2023_01_DET_KC (rows marked kept=false would be dropped):


posteam,passer,dropbacks,epa_sum,epa_per_db,kept
str,str,u32,f64,f64,bool
"""DET""","""J.Goff""",35,3.892088,0.111203,true
"""KC""","""P.Mahomes""",45,-3.528877,-0.078419,true


## Concept 3: outliers

One extreme value can drag an average, like one test wrongly marked 0 sinking a class average.
Two kinds here: **extreme games** (mostly tiny samples: 9 of the 10 most extreme QB-games have <10 dropbacks,
which dropback weights already handle) and **extreme plays** inside normal games (e.g. Cousins' −10.4 pick-six).

In [ ]:
# The 3-dropback cutoff decided in Concept 1 (see DECISIONS.md).
MIN_GAME_DROPBACKS = 3

# Single-play EPA: how spread out is it, and where are the extremes?
train_db = train_pbp.filter(pl.col("qb_dropback") == 1)
print(train_db["epa"].describe())
train_db.sort("epa").select("game_id", "passer", "epa", "desc").head(5)

### ▶ Play: Concept 3
Cap ("clip") every play's EPA to ±`PLAY_EPA_CAP` before averaging, and see which games move.
`None` = no cap (the default target).

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
PLAY_EPA_CAP = None           # Try 6, then 4, then 2. How many plays get clipped? Whose games move most?
SHOW_GAME = "2016_06_PHI_WAS" # Cousins' pick-six game. Try any game_id.
# ---------------------------------------------------------------------------

def capped_games(cap):
    pbp_c = train_pbp if cap is None else train_pbp.with_columns(pl.col("epa").clip(-cap, cap))
    return qb_games(pbp_c).filter(pl.col("dropbacks") >= MIN_GAME_DROPBACKS)

base = capped_games(None)
now = capped_games(PLAY_EPA_CAP)

n_clipped = 0 if PLAY_EPA_CAP is None else train_db.filter(pl.col("epa").abs() > PLAY_EPA_CAP).height
print(f"Plays clipped: {n_clipped:,} of {train_db.height:,} ({n_clipped / train_db.height:.2%})")
print(f"Spread of game EPA/dropback (std): default {base['epa_per_db'].std():.3f}  ->  now {now['epa_per_db'].std():.3f}")

keys = ["game_id", "posteam", "passer_id"]
moved = (base.join(now.select(*keys, pl.col("epa_per_db").alias("now")), on=keys)
             .with_columns((pl.col("now") - pl.col("epa_per_db")).alias("change")))

print("\nStarter games (20+ dropbacks) that moved most:")
print(moved.filter(pl.col("dropbacks") >= 20).sort(pl.col("change").abs(), descending=True)
           .select("game_id", "passer", "dropbacks", "epa_per_db", "now", "change").head(8))

print(f"\nGame {SHOW_GAME}:")
moved.filter(pl.col("game_id") == SHOW_GAME).select("posteam", "passer", "dropbacks", "epa_per_db", "now", "change")

## Concept 4: missing values

A blank is only dangerous when blanks aren't random, like a restaurant survey that unhappy customers skip.
Here `passer_player_id` is blank on every scramble (+0.48 EPA on average vs +0.02 for other dropbacks),
so grouping by it would silently drop the best plays of mobile QBs.

In [ ]:
# Null count per key column, on training dropbacks. A non-zero count is a question to answer:
# *which* rows are blank, and does dropping them bias the target?
key_cols = ["passer_player_id", "passer_id", "passer", "epa", "qb_epa", "cpoe"]
print(train_db.select([pl.col(c).null_count() for c in key_cols]))

# Which kind of dropback has a blank passer_player_id?
(train_db.filter(pl.col("passer_player_id").is_null())
         .group_by("qb_scramble", "sack", "pass_attempt").agg(pl.len().alias("rows")))

### ▶ Play: Concept 4
See what each QB's 2016–2023 EPA/dropback would be if we grouped by a different ID column.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
ID_COLUMN = "passer_id"     # Try "passer_player_id": rows where it's blank drop out. Who loses the most?
MIN_DROPBACKS = 1000        # Only show passers with at least this many train dropbacks. Try 300 to see backups.
# ---------------------------------------------------------------------------

def career(id_col):
    # Rows with a blank ID can't be credited to any QB, so they drop out.
    return (train_db.filter(pl.col(id_col).is_not_null())
                    .group_by(pl.col(id_col).alias("qb_id"))
                    .agg(pl.col("passer").drop_nulls().first(), pl.len().alias("dropbacks"),
                         pl.col("epa").mean().alias("epa_per_db")))

base = career("passer_id")
now = career(ID_COLUMN)
print(f"Dropbacks credited to a QB: default {base['dropbacks'].sum():,}  ->  now {now['dropbacks'].sum():,}")

compare = (base.filter(pl.col("dropbacks") >= MIN_DROPBACKS)
               .join(now.select("qb_id", pl.col("dropbacks").alias("db_now"), pl.col("epa_per_db").alias("now")), on="qb_id")
               .with_columns((pl.col("now") - pl.col("epa_per_db")).alias("change"),
                             (1 - pl.col("db_now") / pl.col("dropbacks")).alias("share_lost")))
compare.sort("change").select("passer", "dropbacks", "share_lost", "epa_per_db", "now", "change").head(10)

## Concept 5: `epa` vs `qb_epa`

`qb_epa` is `epa` with one fix: when a receiver catches the ball and then fumbles it away,
the QB keeps credit for the catch instead of eating the fumble. Like grading a chef on the
dish, not on the waiter who dropped it on the way to the table.

In [ ]:
# On which dropbacks do the two columns disagree? Predict the count before running.
differ = train_db.filter((pl.col("epa") - pl.col("qb_epa")).abs() > 1e-9)
print(f"Dropbacks where epa != qb_epa: {differ.height:,} of {train_db.height:,} ({differ.height / train_db.height:.2%})")

# What kind of play are they? (fumble_lost after a completed catch, or something else?)
print(differ.group_by("complete_pass", "fumble_lost", "sack", "qb_scramble")
            .agg(pl.len().alias("rows"), (pl.col("qb_epa") - pl.col("epa")).mean().alias("avg_qb_epa_minus_epa")))

# The biggest gaps: a short catch, a fumble, a defensive touchdown.
differ.sort(pl.col("qb_epa") - pl.col("epa"), descending=True).select(
    "game_id", "passer", "receiver", "yards_gained", "epa", "qb_epa", "desc").head(5)

### ▶ Play: Concept 5
Switch the per-play column the target is built from, and see which games and careers move.
`"epa"` = the default target.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
EPA_COLUMN = "epa"            # Try "qb_epa". How many games change? Whose career number rises most?
SHOW_GAME = "2019_14_DEN_HOU" # Watson: 8-yard catch, fumble, returned for a TD. Try any game_id.
# ---------------------------------------------------------------------------

def games_with(col):
    # qb_games() reads the "epa" column, so swap the chosen column in under that name.
    return qb_games(train_pbp.with_columns(pl.col(col).alias("epa"))).filter(pl.col("dropbacks") >= MIN_GAME_DROPBACKS)

keys = ["game_id", "posteam", "passer_id"]
moved = (games_with("epa")
         .join(games_with(EPA_COLUMN).select(*keys, pl.col("epa_per_db").alias("now")), on=keys)
         .with_columns((pl.col("now") - pl.col("epa_per_db")).alias("change")))

print(f"QB-games that change: {(moved['change'].abs() > 1e-9).sum():,} of {moved.height:,}")
print(f"Average game EPA/dropback: default {moved['epa_per_db'].mean():+.4f}  ->  now {moved['now'].mean():+.4f}")

print("\nStarter games (20+ dropbacks) that moved most:")
print(moved.filter(pl.col("dropbacks") >= 20).sort("change", descending=True)
           .select("game_id", "passer", "dropbacks", "epa_per_db", "now", "change").head(5))

# Career view (QBs with 1000+ train dropbacks): fumble-prone receivers help some QBs more than others.
career_now = (train_db.with_columns(pl.col(EPA_COLUMN).alias("now"))
              .group_by("passer_id")
              .agg(pl.col("passer").first(), pl.len().alias("dropbacks"),
                   pl.col("epa").mean().alias("epa_per_db"), pl.col("now").mean())
              .filter(pl.col("dropbacks") >= 1000)
              .with_columns((pl.col("now") - pl.col("epa_per_db")).alias("change")))
print("\nCareers that rise most:")
print(career_now.sort("change", descending=True).select("passer", "dropbacks", "epa_per_db", "now", "change").head(5))

print(f"\nGame {SHOW_GAME}:")
moved.filter(pl.col("game_id") == SHOW_GAME).select("posteam", "passer", "dropbacks", "epa_per_db", "now", "change")

## Your exploration

Add cells below.